In [2]:
!pip -q install -U zemberek-python transformers accelerate bitsandbytes huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 MB 9.8 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 117.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 118.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.8 which is incompatible.


In [1]:
import re
import unicodedata
import pandas as pd
from transformers import pipeline
from functools import lru_cache
from collections import defaultdict
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import torch
from functools import lru_cache

@lru_cache(maxsize=1)
def get_punct_pipe():
    model_name = "uygarkurt/electra-restore-punctuation-turkish"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name)

    device = 0 if torch.cuda.is_available() else -1

    return pipeline(
        "token-classification",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="none",
        device=device,
    )

In [2]:
_PUNCT_CHARS = {",", ".", "?", "!", ";", ":"}

def apply_punct(
    text: str,
    pred,
    min_score: float = 0.0,
    prefer=("PERIOD", "QUESTION", "EXCLAMATION", "COMMA", "COLON", "SEMICOLON"),
):
    punct_map = {
        "COMMA": ",", "PERIOD": ".", "QUESTION": "?",
        "EXCLAMATION": "!", "COLON": ":", "SEMICOLON": ";", "O": ""
    }
    prefer_rank = {lab: i for i, lab in enumerate(prefer)}

    # Avoid per-call sort cost (pipeline outputs are typically ordered)
    pred_sorted = pred

    groups = []
    cur = None
    for p in pred_sorted:
        w = p["word"]
        start, end = int(p["start"]), int(p["end"])
        lab = p["entity"]
        score = float(p["score"])
        is_cont = w.startswith("##")

        # CHEAP gap check (no substring/strip)
        if cur is None or (not is_cont and start > cur["end"]):
            cur = {"start": start, "end": end, "best_lab": "O", "best_score": -1.0}
            groups.append(cur)
        else:
            cur["end"] = max(cur["end"], end)

        if lab != "O" and score >= min_score:
            if (score > cur["best_score"]) or (
                score == cur["best_score"]
                and prefer_rank.get(lab, 999) < prefer_rank.get(cur["best_lab"], 999)
            ):
                cur["best_lab"] = lab
                cur["best_score"] = score

    inserts = []
    for g in groups:
        ch = punct_map.get(g["best_lab"], "")
        if not ch:
            continue
        pos = g["end"]
        if pos < len(text) and text[pos:pos+1] in _PUNCT_CHARS:
            continue
        inserts.append((pos, ch))

    if not inserts:
        return text

    inserts.sort()

    out_parts = []
    last = 0
    for pos, ch in inserts:
        if pos < last:
            continue
        out_parts.append(text[last:pos])
        out_parts.append(ch)
        last = pos
    out_parts.append(text[last:])
    return "".join(out_parts)

def _comma_to_period_for_sentences(restored: str, min_words: int = 12) -> str:
    """
    Virgülleri cümle sınırı gibi kullanıp makul uzunlukta parçaları '.' ile kapatır.
    Review gibi akış metinlerde nokta azlığı problemini çözer.
    """
    parts = [p.strip() for p in restored.split(",") if p.strip()]
    out = []
    buf = []
    wc_sum = 0

    for p in parts:
        wc = len(p.split())
        buf.append(p)
        wc_sum += wc

        if wc_sum >= min_words:
            sent = " ".join(buf).strip()
            # zaten .?! ile bitiyorsa tekrar basma
            if not re.search(r"[.!?]$", sent):
                sent += "."
            out.append(sent)
            buf, wc_sum = [], 0

    if buf:
        sent = " ".join(buf).strip()
        if not re.search(r"[.!?]$", sent):
            sent += "."
        out.append(sent)

    return " ".join(out)


def _chunk_by_words(text: str, max_words: int = 160) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    return [" ".join(words[i : i + max_words]) for i in range(0, len(words), max_words)]


def restore_punctuation_text(
    text: str,
    pipe=None,
    max_words_per_chunk: int = 160,
    sentence_mode: bool = False,
    comma_to_period: bool = True,
    comma_min_words: int = 12,
    min_score: float = 0.55,
) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if pipe is None:
        pipe = get_punct_pipe()

    chunks = _chunk_by_words(text, max_words=max_words_per_chunk)
    restored_chunks = []

    prefer = ("PERIOD", "QUESTION", "EXCLAMATION") if sentence_mode else (
        "PERIOD", "QUESTION", "EXCLAMATION", "COMMA", "COLON", "SEMICOLON"
    )

    for ch in chunks:
        pred = pipe(ch)
        restored = apply_punct(ch, pred, min_score=min_score, prefer=prefer)

        # IMPORTANT: no second pass. If comma_to_period=True, use the commas already produced.
        if comma_to_period:
            restored = _comma_to_period_for_sentences(restored, min_words=comma_min_words)

        restored_chunks.append(restored)

    joined = " ".join(restored_chunks).strip()
    joined = re.sub(r"\s+([,.!?;:])", r"\1", joined)
    joined = re.sub(r"\s{2,}", " ", joined)
    return joined


def add_punct_to_col(
    df: pd.DataFrame,
    col: str = "body",
    out_col: str | None = None,
    **restore_kwargs,
) -> pd.DataFrame:
    out = df.copy()
    pipe = get_punct_pipe()
    target = out_col or col

    # ---- Batch the entire column instead of pandas.apply ----
    texts = out[col].fillna("").astype(str).tolist()

    max_words_per_chunk = restore_kwargs.get("max_words_per_chunk", 160)
    min_score = restore_kwargs.get("min_score", 0.55)
    sentence_mode = restore_kwargs.get("sentence_mode", False)
    comma_to_period = restore_kwargs.get("comma_to_period", True)
    comma_min_words = restore_kwargs.get("comma_min_words", 12)

    prefer = ("PERIOD", "QUESTION", "EXCLAMATION") if sentence_mode else (
        "PERIOD", "QUESTION", "EXCLAMATION", "COMMA", "COLON", "SEMICOLON"
    )

    # 1) Flatten row texts into chunk tasks
    tasks = []  # (row_i, chunk_i, chunk_text)
    for i, t in enumerate(texts):
        t = "" if t is None else str(t).strip()
        if not t:
            continue
        chunks = _chunk_by_words(t, max_words=max_words_per_chunk)
        for ci, ch in enumerate(chunks):
            tasks.append((i, ci, ch))

    if not tasks:
        out[target] = [""] * len(texts)
        return out

    chunk_texts = [t[2] for t in tasks]

    # 2) GPU inference in batches (tune batch_size)
    batch_size = restore_kwargs.get("batch_size", 32)
    
    with torch.inference_mode():
        preds = pipe(chunk_texts, batch_size=batch_size)

    # 3) Postprocess and reassemble
    per_row = defaultdict(list)
    for (row_i, ci, ch_text), pred in zip(tasks, preds):
        restored = apply_punct(ch_text, pred, min_score=min_score, prefer=prefer)
        if comma_to_period:
            restored = _comma_to_period_for_sentences(restored, min_words=comma_min_words)
        restored = re.sub(r"\s+([,.!?;:])", r"\1", restored)
        restored = re.sub(r"\s{2,}", " ", restored).strip()
        per_row[row_i].append((ci, restored))

    out_col_vals = [""] * len(texts)
    for i in range(len(texts)):
        chunks = per_row.get(i)
        if not chunks:
            out_col_vals[i] = ""
        else:
            chunks.sort(key=lambda x: x[0])
            out_col_vals[i] = " ".join(c for _, c in chunks).strip()

    # after building tasks
    print("rows:", len(texts))
    print("tasks (row,chunk):", len(tasks))
    
    chunk_texts = [t[2] for t in tasks]
    print("chunk_texts:", len(chunk_texts))

    out[target] = out_col_vals
    return out

In [3]:
# URL + email patterns
_URL_RE = re.compile(r"""(?ix)
\b(
  https?://[^\s<>()]+ |
  www\.[^\s<>()]+ |
  [a-z0-9.-]+\.(com|net|org|io|co|tr|de|uk|me|app|shop|store|gov|edu)(/[^\s<>()]*)?
)\b
""")

_EMAIL_RE = re.compile(r"""(?ix)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b""")

# Invisible / zero-width chars
_INVISIBLE_RE = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")

# Whitespace normalization
_MULTI_SPACE_RE = re.compile(r"[ \t]+")
_MULTI_NL_RE = re.compile(r"\n{3,}")

# Repeated punctuation
_REPEAT_EXCL_RE = re.compile(r"!{2,}")
_REPEAT_Q_RE = re.compile(r"\?{2,}")
_ELLIPSIS_RE = re.compile(r"(?:\.{2,}|…{2,})")  # 2+ dots or ellipsis -> single ellipsis

# Spacing around punctuation
_SPACE_BEFORE_PUNCT_RE = re.compile(r"\s+([,;:.!?…])")
_SPACE_AFTER_OPEN_RE = re.compile(r"([\(\[\{«“])\s+")
_SPACE_BEFORE_CLOSE_RE = re.compile(r"\s+([\)\]\}»”])")

# Insert missing space AFTER sentence-final punctuation, but not for decimals/thousands (handled below)
_NO_SPACE_AFTER_PUNCT_RE = re.compile(r"([.!?…])([A-Za-zÇĞİÖŞÜçğıöşü0-9\(\[\{“\"'])")

# Protect dots inside numbers like 3.5 or 1.000 before punctuation spacing
_DECIMAL_DOT_RE = re.compile(r"(?<=\d)\.(?=\d)")          # 3.5
_THOUSAND_DOT_RE = re.compile(r"(?<=\d)\.(?=\d{3}\b)")    # 1.000

_DOT_TOKEN = "<DOT>"

In [4]:
def _protect_number_dots(text: str) -> str:
    text = _DECIMAL_DOT_RE.sub(_DOT_TOKEN, text)
    text = _THOUSAND_DOT_RE.sub(_DOT_TOKEN, text)
    return text

def _restore_number_dots(text: str) -> str:
    return text.replace(_DOT_TOKEN, ".")

def clean_review_text(
    text,
    mask_urls: bool = True,
    mask_emails: bool = True,
    normalize_unicode: bool = True,
    normalize_ellipsis: bool = True,
    reduce_repeated_punct: bool = True,
    fix_punct_spacing: bool = True,
) -> str:
    """
    Produces a cleaned review string suitable for sentence segmentation
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    text = str(text)

    # Normalize newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Strip invisible chars
    text = _INVISIBLE_RE.sub("", text)

    # Unicode normalize (quotes, full-width chars, etc.)
    if normalize_unicode:
        text = unicodedata.normalize("NFKC", text)

    # Mask URLs / emails
    if mask_urls:
        text = _URL_RE.sub("<URL>", text)
    if mask_emails:
        text = _EMAIL_RE.sub("<EMAIL>", text)

    # Normalize ellipsis / repeated punctuation
    if normalize_ellipsis:
        text = _ELLIPSIS_RE.sub("…", text)

    if reduce_repeated_punct:
        text = _REPEAT_EXCL_RE.sub("!", text)
        text = _REPEAT_Q_RE.sub("?", text)

    # Whitespace normalize
    text = _MULTI_SPACE_RE.sub(" ", text)
    text = _MULTI_NL_RE.sub("\n\n", text)
    text = text.strip()

    # conservative + number-safe
    if fix_punct_spacing and text:
        text = _protect_number_dots(text)

        text = _SPACE_BEFORE_PUNCT_RE.sub(r"\1", text)
        text = _SPACE_AFTER_OPEN_RE.sub(r"\1", text)
        text = _SPACE_BEFORE_CLOSE_RE.sub(r"\1", text)
        text = _NO_SPACE_AFTER_PUNCT_RE.sub(r"\1 \2", text)

        text = _restore_number_dots(text)

        # Final trim after edits
        text = text.strip()

    return text

def add_clean_body(df: pd.DataFrame, text_col: str = "body") -> pd.DataFrame:
    out = df.copy()
    out["body_raw"] = out[text_col]
    out["clean_body"] = out[text_col].apply(clean_review_text)
    return out

In [5]:
def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out = add_punct_to_col(
        out,
        col="body",
        out_col="punctuation_restored_body",
        sentence_mode=False,
        comma_to_period=True,
        comma_min_words=6,
        min_score=0.55,
        max_words_per_chunk=120,  # safer than 160 for token limits
        batch_size=8,            # BIG speedup on T4; try 16/32/64
    )

    out = add_clean_body(out, text_col="punctuation_restored_body")
    return out

In [6]:
import pandas as pd
from zemberek import TurkishSentenceExtractor

ASPECT_COLS = ["kargo", "paketleme", "fiyat_performans", "kalite", "genel_memnuniyet"]

def build_reviews_and_sentences(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    reviews_df = df.copy()
    reviews_df = reviews_df.reset_index().rename(columns={"index": "review_id"})

    extractor = TurkishSentenceExtractor()
    sentence_rows = []

    for row in reviews_df.itertuples(index=False):
        review_id = row.review_id
        text = str(getattr(row, "clean_body", "") or "").strip()
        if not text:
            continue

        sents = [s.strip() for s in extractor.from_paragraph(text) if s and s.strip()]
        n = len(sents)
        if n == 0:
            continue

        # metadata
        source = getattr(row, "source", None)
        rating = getattr(row, "rating", None)
        review_date = getattr(row, "review_date", None)
        collected_at = getattr(row, "collected_at", None)
        product_title = getattr(row, "product_title", None)
        product_url = getattr(row, "product_url", None)
        title = getattr(row, "title", None)
        review_summary = getattr(row, "review_summary", None)

        for i, sent in enumerate(sents, start=1):
            sentence_rows.append({
                "review_id": review_id,
                "sent_idx": i,
                "n_sentences": n,

                "sentence_clean": sent,
                "sentence_text_variant": "clean_body",

                "source": source,
                "rating": rating,
                "review_date": review_date,
                "collected_at": collected_at,
                "product_title": product_title,
                "product_url": product_url,
                "title": title,
                "review_summary": review_summary,
            })

    sentences_df = pd.DataFrame(sentence_rows)

    # enforce dtypes for labels (0/1/NA)
    for c in ASPECT_COLS:
        if c in sentences_df.columns:
            sentences_df[c] = sentences_df[c].astype("Int8")

    return reviews_df, sentences_df

In [7]:
path = r"C:\Users\berrk\OneDrive\Belgeler\GitHub\Aspect-Aware-Ecommerce-Summarizer\output.csv"
df = pd.read_csv(path)
df = preprocess_df(df)

reviews_df, sentences_df = build_reviews_and_sentences(df)

print(reviews_df.shape, sentences_df.shape)
sentences_df.head()

Device set to use cpu


rows: 35
tasks (row,chunk): 35
chunk_texts: 35
(35, 16) (181, 13)


,review_id,sent_idx,n_sentences,sentence_clean,sentence_text_variant,source,rating,review_date,collected_at,product_title,product_url,title,review_summary
0,0,1,1,1 yıldır kullanıyoruz bu 3. Tavsiye ürün herha...,clean_body,amazon_tr,5.0,2026-04-14,2026-04-17 17:43,Philips HomeRun 2000 Robot Süpürge ve Paspas -...,https://www.amazon.com.tr/dp/B0FVYFD3HZ,1 yıldır kullanıyoruz bu 3. Tavsiye ürün,NaN
1,1,1,15,Ürünü kullanmaya başlar başlamaz en çok etkile...,clean_body,amazon_tr,5.0,2026-04-09,2026-04-17 17:43,Philips HomeRun 2000 Robot Süpürge ve Paspas -...,https://www.amazon.com.tr/dp/B0FVYFD3HZ,Philips HomeRun 2000 Robot Süpürge,NaN
2,1,2,15,Evi çok kısa sürede detaylı ve doğru şekilde t...,clean_body,amazon_tr,5.0,2026-04-09,2026-04-17 17:43,Philips HomeRun 2000 Robot Süpürge ve Paspas -...,https://www.amazon.com.tr/dp/B0FVYFD3HZ,Philips HomeRun 2000 Robot Süpürge,NaN
3,1,3,15,neredeyse hiç şaşırmıyor.,clean_body,amazon_tr,5.0,2026-04-09,2026-04-17 17:43,Philips HomeRun 2000 Robot Süpürge ve Paspas -...,https://www.amazon.com.tr/dp/B0FVYFD3HZ,Philips HomeRun 2000 Robot Süpürge,NaN
4,1,4,15,Harita üzerinde oda seçmek.,clean_body,amazon_tr,5.0,2026-04-09,2026-04-17 17:43,Philips HomeRun 2000 Robot Süpürge ve Paspas -...,https://www.amazon.com.tr/dp/B0FVYFD3HZ,Philips HomeRun 2000 Robot Süpürge,NaN


## Label New DF ( LOAD MODEL )

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class AspectLabelingPipeline:
    def __init__(self, model_dir, text_col="sentence_clean", batch_size=64, device=None):
        self.model_dir = model_dir
        self.text_col = text_col
        self.batch_size = batch_size

        # device
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        # load config
        with open(os.path.join(model_dir, "inference_config.json"), "r", encoding="utf-8") as f:
            cfg = json.load(f)

        self.labels = cfg["labels"]
        self.max_len = cfg["max_len"]
        self.label2id = {l: i for i, l in enumerate(self.labels)}
        self.id2label = {i: l for i, l in enumerate(self.labels)}

        # load thresholds
        with open(os.path.join(model_dir, "thresholds.json"), "r", encoding="utf-8") as f:
            th = json.load(f)

        self.thresholds = np.array(th["thresholds"], dtype=np.float32)

        # load tokenizer + model
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()

        # lightweight Trainer for batched predict
        self.trainer = Trainer(model=self.model)

    def _prepare_dataframe(self, df):
        out = df.copy()
        out[self.text_col] = out[self.text_col].fillna("").astype(str).str.strip()
        return out

    def _make_dataset(self, df):
        ds = Dataset.from_pandas(
            df[[self.text_col]].rename(columns={self.text_col: "text"}),
            preserve_index=False
        )

        def tokenize(batch):
            return self.tokenizer(
                batch["text"],
                truncation=True,
                padding="max_length",
                max_length=self.max_len
            )

        ds = ds.map(tokenize, batched=True, remove_columns=["text"])
        return ds

    def predict_proba(self, df):
        df = self._prepare_dataframe(df)
        ds = self._make_dataset(df)

        pred = self.trainer.predict(ds)
        logits = pred.predictions
        probs = sigmoid(logits)
        return probs

    def predict(self, df, return_probs=True, return_binary=True):
        df_out = self._prepare_dataframe(df)
        probs = self.predict_proba(df_out)

        # apply tuned thresholds
        y_pred = (probs >= self.thresholds.reshape(1, -1)).astype(int)

        # convert to label lists
        pred_labels = []
        for row in y_pred:
            labs = [self.labels[j] for j, v in enumerate(row) if v == 1]
            pred_labels.append(labs)

        df_out["predicted_labels"] = pred_labels

        if return_binary:
            for j, lab in enumerate(self.labels):
                df_out[f"pred_{lab}"] = y_pred[:, j]

        if return_probs:
            for j, lab in enumerate(self.labels):
                df_out[f"prob_{lab}"] = probs[:, j]

        return df_out

In [3]:
sentences_df = pd.read_csv("/kaggle/input/datasets/berkhanguzeller/aspect-summary/sentence_clean (1500 urun).csv")

In [4]:
MODEL_DIR = "/kaggle/input/models/berkhanguzeller/aspect-labeler-v1/other/default/1"

pipe = AspectLabelingPipeline(
    model_dir=MODEL_DIR,
    text_col="sentence_clean",
    batch_size=64
)

labeled_df = pipe.predict(sentences_df, return_probs=True, return_binary=True)
labeled_df.head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/259207 [00:00<?, ? examples/s]

,Unnamed: 0,review_id,sent_idx,n_sentences,sentence_clean,sentence_text_variant,source,rating,review_date,collected_at,...,pred_kargo,pred_paketleme,pred_fiyat_performans,pred_kalite,pred_genel_memnuniyet,prob_kargo,prob_paketleme,prob_fiyat_performans,prob_kalite,prob_genel_memnuniyet
0,0,0,1,5,ürün fiyatına göre gayet iyi.,clean_body,amazon_tr,5.0,2026-03-03,2026-03-19 11:22,...,0,0,1,0,1,0.026430,0.028281,0.998288,0.118822,0.703205
1,1,0,2,5,ses çıkartmayacağını düşünerek almayın.,clean_body,amazon_tr,5.0,2026-03-03,2026-03-19 11:22,...,0,0,0,1,0,0.001564,0.004046,0.022276,0.810865,0.323051
2,2,0,3,5,çünkü böyle bir özelliği yok.,clean_body,amazon_tr,5.0,2026-03-03,2026-03-19 11:22,...,0,0,0,1,0,0.007064,0.007965,0.006170,0.952064,0.014553
3,3,0,4,5,diğer mouse'lar gibi.,clean_body,amazon_tr,5.0,2026-03-03,2026-03-19 11:22,...,0,0,0,1,0,0.005228,0.006051,0.006906,0.891236,0.041572
4,4,0,5,5,onun dışında oyun vs oynamayacaksanız gayet uy...,clean_body,amazon_tr,5.0,2026-03-03,2026-03-19 11:22,...,0,0,1,0,0,0.030447,0.028327,0.998667,0.338781,0.382927


In [7]:
labeled_df[["sentence_clean", "predicted_labels"]].tail()

,sentence_clean,predicted_labels
259202,Şöyle mistik.,[]
259203,böyle güzel denen sabun anneanne evinde büfede...,"[kalite, genel_memnuniyet]"
259204,Siz siz olun.,[genel_memnuniyet]
259205,koklamadan almayın…,[genel_memnuniyet]
259206,gerçekten tekrar tekrar el yıkama isteği oluşt...,"[kalite, genel_memnuniyet]"


In [8]:
labeled_df.to_csv("Labeled Dataframe (1500 urun).csv")

## Embedding Pipeline

In [9]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# ── CONFIG ─────────────────────────────────────────────────────────────────

TOP_K_CANDIDATES = 100   # MMR'a girecek aday havuzu
TOP_K_FINAL      = 30    # MMR'dan çıkacak final cümle sayısı
MMR_LAMBDA       = 0.7   # 1.0 = sadece relevance, 0.0 = sadece diversity
DUP_THRESHOLD    = 0.85  # bu üzeri = duplicate say
MIN_TOKENS       = 4     # bu altı = gürültü
MIN_RELEVANCE = 0.1

df = labeled_df.copy()

ASPECTS = ["kargo", "paketleme", "fiyat_performans", "kalite", "genel_memnuniyet"]
PRED_COLS = {a: f"pred_{a}" for a in ASPECTS}

ASPECT_QUERIES = {
    "kargo":            "kargo hızı teslimat süresi paket geldi",
    "paketleme":        "paket ambalaj kutu hasar koruma",
    "fiyat_performans": "fiyat performans değer para eder mi",
    "kalite":           "ürün kalitesi malzeme sağlamlık dayanıklılık",
    "genel_memnuniyet": "genel memnuniyet tavsiye eder tekrar alır",
}

In [10]:
# ── VERİYİ OKU ─────────────────────────────────────────────────────────────

df["product_id"] = df["product_url"].str.extract(r'/dp/([A-Z0-9]{10})')
df["product_id"] = df["product_id"].fillna(
    df["product_title"].str.slice(0, 50).str.strip()
)
df = df.reset_index(drop=True)

In [11]:
# ── GÜRÜLTÜ TEMİZLİĞİ ──────────────────────────────────────────────────────

def is_noisy(text: str) -> bool:
    if not isinstance(text, str):
        return True
    tokens = text.strip().split()
    if len(tokens) < MIN_TOKENS:
        return True
    # Sadece emoji / noktalama / ünlem
    clean = text.strip("!?. \n\t")
    if len(clean) == 0:
        return True
    return False

df["is_noisy"] = df["sentence_clean"].apply(is_noisy)
print(f"Gürültülü cümle: {df['is_noisy'].sum()} / {len(df)}")
df = df[~df["is_noisy"]].reset_index(drop=True)

Gürültülü cümle: 70537 / 259207


In [12]:
# ── ENCODE ─────────────────────────────────────────────────────────────────

model = SentenceTransformer("trmteb/turkish-embedding-model") #SentenceTransformer("emrecan/bert-base-turkish-cased-mean-nli-stsb-tr")
#TurkEmbed4STS

print("Cümleler encode ediliyor...")
all_embeddings = model.encode(
    df["sentence_clean"].tolist(),
    batch_size=32, #256
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

aspect_vecs = {
    aspect: model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    for aspect, query in ASPECT_QUERIES.items()
}

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Cümleler encode ediliyor...


Batches:   0%|          | 0/5896 [00:00<?, ?it/s]

In [13]:
# ── MMR ────────────────────────────────────────────────────────────────────

def mmr(candidate_vecs: np.ndarray,
        query_vec: np.ndarray,
        top_k: int,
        lam: float = 0.7) -> list[int]:
    """
    Maximal Marginal Relevance.
    Döndürür: candidate_vecs içindeki local index listesi.
    """
    relevance = candidate_vecs @ query_vec          # (M,)
    selected  = []
    remaining = list(range(len(candidate_vecs)))

    while len(selected) < top_k and remaining:
        if not selected:
            # İlk adımda sadece relevance'a bak
            best = max(remaining, key=lambda i: relevance[i])
        else:
            sel_vecs = candidate_vecs[selected]     # (S, D)
            best_score = -np.inf
            best = None
            for i in remaining:
                rel  = relevance[i]
                # Seçilmiş cümlelere max benzerlik
                max_sim = float((candidate_vecs[i] @ sel_vecs.T).max())
                score = lam * rel - (1 - lam) * max_sim
                if score > best_score:
                    best_score = score
                    best = i
        selected.append(best)
        remaining.remove(best)

    return selected

In [14]:
# ── DATASET OLUŞTUR ─────────────────────────────────────────────────────────

records = []

for product_id, grp in df.groupby("product_id"):
    for aspect in ASPECTS:
        pred_col = PRED_COLS[aspect]

        subset = grp[grp[pred_col] == 1].copy()
        if subset.empty:
            continue

        idxs = subset.index.values
        candidate_vecs = all_embeddings[idxs]
        query_vec      = aspect_vecs[aspect]

        # 1. Relevance filtresi
        relevance_scores = candidate_vecs @ query_vec
        relevance_mask   = relevance_scores >= MIN_RELEVANCE
        if relevance_mask.sum() == 0:
            continue
        
        idxs             = idxs[relevance_mask]
        candidate_vecs   = candidate_vecs[relevance_mask]
        relevance_scores = relevance_scores[relevance_mask]
        
        # 2. Near-duplicate temizliği — relevance'a göre sıralı gezerek
        order      = np.argsort(relevance_scores)[::-1]
        keep_mask  = np.ones(len(idxs), dtype=bool)
        
        for i_pos, i in enumerate(order):
            if not keep_mask[i]:
                continue
            for j in order[i_pos + 1:]:
                if not keep_mask[j]:
                    continue
                sim = float(candidate_vecs[i] @ candidate_vecs[j])
                if sim >= DUP_THRESHOLD:
                    keep_mask[j] = False  # daha düşük relevance'lıyı at
        
        idxs             = idxs[keep_mask]
        candidate_vecs   = candidate_vecs[keep_mask]
        relevance_scores = relevance_scores[keep_mask]
        
        # 3. TOP_K_CANDIDATES → MMR
        top_candidate_local = np.argsort(relevance_scores)[::-1][:TOP_K_CANDIDATES]

        candidate_vecs_top = candidate_vecs[top_candidate_local]
        idxs_top           = idxs[top_candidate_local]

        # MMR uygula
        selected_local = mmr(
            candidate_vecs=candidate_vecs_top,
            query_vec=query_vec,
            top_k=TOP_K_FINAL,
            lam=MMR_LAMBDA,
        )

        for rank, local_i in enumerate(selected_local):
            global_i = idxs_top[local_i]
            row      = df.loc[global_i]
            rel_score = float(relevance_scores[top_candidate_local[local_i]])

            records.append({
                "product_id":       product_id,
                "product_title":    row["product_title"],
                "aspect":           aspect,
                "sentence":         row["sentence_clean"],
                "relevance_score":  round(rel_score, 4),
                "mmr_rank":         rank + 1,
                "rating":           row["rating"],
                "review_id":        row["review_id"],
                "sent_idx":         row["sent_idx"],
                "source":           row["source"],
            })

out = pd.DataFrame(records)

In [15]:
# ── İSTATİSTİKLER ──────────────────────────────────────────────────────────

print(f"\n{'='*45}")
print(f"Toplam kayıt:  {len(out)}")
print(f"Unique ürün:   {out['product_id'].nunique()}")
print(f"\nAspect başına kayıt:")
print(out.groupby("aspect")["sentence"].count().to_string())
print(f"\nÜrün x aspect başına cümle sayısı:")
print(out.groupby(["product_id","aspect"])["sentence"].count().describe().round(1).to_string())

# ── KAYDET ─────────────────────────────────────────────────────────────────

out.to_parquet("aspect_sentences_mmr.parquet", index=False)
out.to_csv("aspect_sentences_mmr.csv", index=False)
print("\nKaydedildi → aspect_sentences_mmr.parquet / .csv")


Toplam kayıt:  100452
Unique ürün:   1538

Aspect başına kayıt:
aspect
fiyat_performans    13633
genel_memnuniyet    29974
kalite              35398
kargo               13117
paketleme            8330

Ürün x aspect başına cümle sayısı:
count    7020.0
mean       14.3
std        10.6
min         1.0
25%         5.0
50%        11.0
75%        26.0
max        30.0

Kaydedildi → aspect_sentences_mmr.parquet / .csv


In [16]:
# ── ÖRNEK ──────────────────────────────────────────────────────────────────

sample_pid = out["product_id"].iloc[0]
print(f"\nÖrnek — ilk ürünün 'kalite' cümleleri (MMR sırası):")
print(
    out[(out["product_id"] == sample_pid) & (out["aspect"] == "kalite")]
    [["mmr_rank", "sentence", "relevance_score"]]
    .to_string(index=False)
)


Örnek — ilk ürünün 'kalite' cümleleri (MMR sırası):
 mmr_rank                                                                                                      sentence  relevance_score
        1                                                                                    Ürün kaliteli ve başarılı…           0.6092
        2                                               %100 orijinal eski kalınlığında değil ama renk olarak sorunsuz…           0.3971
        3                                                    Makineye birebir uyumlu ve açıklamalarda anlatıldığı gibi.           0.3082
        4                                                                 Amazon’dan aldığım son derece kaliteli çıktı…           0.3898
        5                                                                    başka kullanmadim kalitesine diyecek yok💯.           0.2900
        6                                                                    Son kullanma tarihleri uzun süreli olarak.      

In [25]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

ASPECTS = ["kargo", "paketleme", "fiyat_performans", "kalite", "genel_memnuniyet"]

TOP_N_PER_ASPECT = 15
MIN_SENTENCES_PER_ASPECT = 1
MAX_SENT_LEN = 300

def clean_sentence_for_export(text: str, max_len: int = MAX_SENT_LEN) -> str:
    if not isinstance(text, str):
        return ""
    text = " ".join(text.strip().split())
    if not text:
        return ""
    return text[:max_len].strip()

def prepare_product_level_dataset(
    retrieval_df: pd.DataFrame,
    top_n_per_aspect: int = TOP_N_PER_ASPECT,
    min_sentences_per_aspect: int = MIN_SENTENCES_PER_ASPECT
) -> pd.DataFrame:
    """
    retrieval_df beklenen kolonlar:
    - product_id
    - product_title
    - aspect
    - sentence
    - mmr_rank
    - relevance_score
    - rating (opsiyonel)
    - review_id (opsiyonel)
    - sent_idx (opsiyonel)
    - source (opsiyonel)
    """

    required_cols = {"product_id", "product_title", "aspect", "sentence", "mmr_rank", "relevance_score"}
    missing = required_cols - set(retrieval_df.columns)
    if missing:
        raise ValueError(f"Eksik kolonlar var: {missing}")

    df = retrieval_df.copy()

    # Temizlik
    df["sentence"] = df["sentence"].apply(clean_sentence_for_export)
    df = df[df["sentence"].str.len() > 0].copy()

    # En güvenli sıralama
    df = df.sort_values(
        ["product_id", "aspect", "mmr_rank", "relevance_score"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)

    product_rows = []

    for product_id, grp in df.groupby("product_id", sort=False):
        title_series = (
            grp["product_title"]
            .dropna()
            .astype(str)
            .str.strip()
        )
        title_series = title_series[title_series != ""]
    
        product_title = title_series.iloc[0] if not title_series.empty else ""
    
        evidence = {}
        evidence_meta = {}
        total_sentences = 0

        for aspect in ASPECTS:
            sub = grp[grp["aspect"] == aspect].copy()
            sub = sub.sort_values(["mmr_rank", "relevance_score"], ascending=[True, False]).head(top_n_per_aspect)

            sentences = sub["sentence"].tolist()
            sentences = [s for s in sentences if isinstance(s, str) and s.strip()]

            if len(sentences) < min_sentences_per_aspect:
                evidence[aspect] = []
                evidence_meta[aspect] = {
                    "n_sentences": 0,
                    "avg_relevance": None,
                    "max_relevance": None,
                }
            else:
                evidence[aspect] = sentences
                evidence_meta[aspect] = {
                    "n_sentences": len(sentences),
                    "avg_relevance": round(float(sub["relevance_score"].mean()), 4) if len(sub) else None,
                    "max_relevance": round(float(sub["relevance_score"].max()), 4) if len(sub) else None,
                }
                total_sentences += len(sentences)

        product_rows.append({
            "product_id": product_id,
            "product_title": product_title,
            "total_evidence_sentences": total_sentences,
            "evidence": evidence,
            "evidence_meta": evidence_meta,
        })

    return pd.DataFrame(product_rows)

In [18]:
def render_aspect_block(evidence_dict: dict) -> str:
    lines = []

    for aspect in ASPECTS:
        lines.append(f"[{aspect}]")
        sents = evidence_dict.get(aspect, [])

        if not sents:
            lines.append("-")
        else:
            for i, sent in enumerate(sents, 1):
                lines.append(f"{i}. {sent}")
        lines.append("")

    return "\n".join(lines).strip()

def build_llm_input_text(product_id: str, product_title: str, evidence: dict) -> str:
    aspect_block = render_aspect_block(evidence)

    return f"""Below are aspect-grouped user review sentences for a single product.

Product ID: {product_id}
Product title: {product_title}

Aspect evidence:
{aspect_block}

Important:
- Summarize the dominant supported view for each aspect.
- Ignore minor opposing evidence.
- Do not balance the sentence artificially.
- Do not use contrast words unless the evidence is strongly and genuinely mixed.
- Keep each aspect strictly within its own meaning.
- Return only valid JSON.
"""

In [19]:
def add_input_text_column(product_df: pd.DataFrame) -> pd.DataFrame:
    df = product_df.copy()

    df["input_text"] = df.apply(
        lambda row: build_llm_input_text(
            product_id=row["product_id"],
            product_title=row["product_title"],
            evidence=row["evidence"]
        ),
        axis=1
    )
    return df

In [20]:
def save_product_dataset_jsonl(product_df: pd.DataFrame, output_path: str = "product_level_dataset.jsonl"):
    output_path = Path(output_path)

    with output_path.open("w", encoding="utf-8") as f:
        for _, row in product_df.iterrows():
            obj = {
                "product_id": row["product_id"],
                "product_title": row["product_title"],
                "total_evidence_sentences": int(row["total_evidence_sentences"]),
                "evidence": row["evidence"],
                "input_text": row["input_text"],
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    print(f"Kaydedildi: {output_path.resolve()}")

In [21]:
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "aspect_summary_output",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "aspect_summaries": {
                    "type": "object",
                    "properties": {
                        "kargo": {"type": ["string", "null"]},
                        "paketleme": {"type": ["string", "null"]},
                        "fiyat_performans": {"type": ["string", "null"]},
                        "kalite": {"type": ["string", "null"]},
                        "genel_memnuniyet": {"type": ["string", "null"]}
                    },
                    "required": [
                        "kargo",
                        "paketleme",
                        "fiyat_performans",
                        "kalite",
                        "genel_memnuniyet"
                    ],
                    "additionalProperties": False
                }
            },
            "required": ["aspect_summaries"],
            "additionalProperties": False
        }
    }
}

In [28]:
SYSTEM_PROMPT = """You summarize Turkish e-commerce reviews by aspect.

Task:
- You will receive review evidence for one product, grouped by aspect.
- Write exactly one short Turkish sentence for each aspect.
- Use only the evidence for that aspect.

Aspect meanings:
- kargo: Kargo süreci, teslimat hızı, kurye deneyimi ve gönderim performansı.
- paketleme: Ambalaj kalitesi, kutu/paket durumu, ürünün korunması ve hasar görüp görmediği.
- fiyat_performans: Ürünün fiyatına değip değmediği, para-değer dengesi ve fiyat avantajı.
- kalite: Ürünün malzeme kalitesi, sağlamlığı, dayanıklılığı ve işçiliği.
- genel_memnuniyet: Genel kullanıcı memnuniyeti, tavsiye edilip edilmediği ve tekrar satın alma eğilimi.

Core rules:
1. Output language: Turkish.
2. Maximum one sentence per aspect.
3. If evidence is weak, missing, off-topic, or insufficient, return null for that aspect.
4. Use the aspect meaning as a strict boundary.
5. Do not make numerical or quantity-based claims.
6. Do not use words like "çoğu", "birçok", "bazı", "kimi", "genelde", "çoğunlukla", or similar frequency wording.
7. Do not automatically mention both positive and negative evidence.
8. If one side is clearly dominant, write only the dominant side.
9. Do not mention minority, isolated, or one-off opposing opinions.
10. Only write a mixed summary if the evidence is strongly and clearly balanced.
11. Do not use contrast words like "ama", "ancak", "buna karşın", "öte yandan" unless the evidence is strongly balanced.
12. Keep aspects semantically pure.
13. Do not copy review text verbatim.
14. Return only the structured output that matches the provided schema."""

In [29]:
def save_openai_requests_jsonl(
    product_df: pd.DataFrame,
    output_path: str = "teacher_requests.jsonl",
    model: str = "gpt-5.4-mini",
    temperature: float = 0.2
):
    output_path = Path(output_path)

    with output_path.open("w", encoding="utf-8") as f:
        for _, row in product_df.iterrows():
            req = {
                "custom_id": str(row["product_id"]),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,
                    "temperature": temperature,
                    "response_format": RESPONSE_SCHEMA,
                    "messages": [
                        {"role": "developer", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": row["input_text"]},
                    ]
                }
            }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

In [30]:
product_df = prepare_product_level_dataset(
    retrieval_df=out,
    top_n_per_aspect=20,
    min_sentences_per_aspect=1
)

product_df = add_input_text_column(product_df)

# product_df.to_pickle("product_level_dataset.pkl")
# product_df.to_json("product_level_dataset.json", orient="records", force_ascii=False, indent=2)

save_product_dataset_jsonl(product_df, "product_level_dataset.jsonl")
save_openai_requests_jsonl(
    product_df,
    output_path="teacher_requests.jsonl",
    model="gpt-5.4-mini",
    temperature=0.2
)

print("\nHazır.")
print("product_level_dataset.jsonl ve teacher_requests.jsonl kaydedildi.")
print("Toplam ürün:", len(product_df))

Kaydedildi: /kaggle/working/product_level_dataset.jsonl

Hazır.
product_level_dataset.jsonl ve teacher_requests.jsonl kaydedildi.
Toplam ürün: 1538


In [31]:
import json

path = "/kaggle/working/teacher_requests.jsonl"

with open(path, "r", encoding="utf-8") as f:
    first = json.loads(next(f))

print(json.dumps(first, ensure_ascii=False, indent=2))

{
  "custom_id": "B0000C73CQ",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-5.4-mini",
    "temperature": 0.2,
    "response_format": {
      "type": "json_schema",
      "json_schema": {
        "name": "aspect_summary_output",
        "strict": true,
        "schema": {
          "type": "object",
          "properties": {
            "aspect_summaries": {
              "type": "object",
              "properties": {
                "kargo": {
                  "type": [
                    "string",
                    "null"
                  ]
                },
                "paketleme": {
                  "type": [
                    "string",
                    "null"
                  ]
                },
                "fiyat_performans": {
                  "type": [
                    "string",
                    "null"
                  ]
                },
                "kalite": {
                  "type": [
               

In [ ]:
import os
import json
from openai import OpenAI

# 1) API key
client = OpenAI()

BATCH_INPUT_FILE = "/kaggle/working/teacher_requests.jsonl"
BATCH_ID_SAVE_PATH = "/kaggle/working/batch_id.txt"

# 2. Upload the JSONL file
uploaded_file = client.files.create(
    file=open(BATCH_INPUT_FILE, "rb"),
    purpose="batch"
)

print("Uploaded file id:", uploaded_file.id)

# 3. Create the batch job
batch = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "job_name": "teacher_aspect_summary_batch"
    }
)

print("Batch created successfully")
print("Batch ID:", batch.id)
print("Status:", batch.status)

# 4. Save batch id so you can check it later
with open(BATCH_ID_SAVE_PATH, "w", encoding="utf-8") as f:
    f.write(batch.id)

print(f"Batch ID saved to: {BATCH_ID_SAVE_PATH}")

Uploaded file id: file-PFRXQq64cs8x5bPjodndQx
Batch created successfully
Batch ID: batch_69c7cd1e7d2c8190abef07e0ea82039e
Status: validating
Batch ID saved to: /kaggle/working/batch_id.txt


In [42]:
batch = client.batches.retrieve("batch_69c7cd1e7d2c8190abef07e0ea82039e")
print(batch)

Batch(id='batch_69c7cd1e7d2c8190abef07e0ea82039e', completion_window='24h', created_at=1774701854, endpoint='/v1/chat/completions', input_file_id='file-PFRXQq64cs8x5bPjodndQx', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1774702281, error_file_id=None, errors=None, expired_at=None, expires_at=1774788254, failed_at=None, finalizing_at=1774702159, in_progress_at=1774701918, metadata={'job_name': 'teacher_aspect_summary_batch'}, model='gpt-5.4-mini-2026-03-17', output_file_id='file-EpzSw8xF8Ud2S9SAvAKKNq', request_counts=BatchRequestCounts(completed=1538, failed=0, total=1538), usage=BatchUsage(input_tokens=2328049, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=178010, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2506059))


In [43]:
output_file_id = "file-EpzSw8xF8Ud2S9SAvAKKNq"
save_path = "/kaggle/working/teacher_batch_output.jsonl"

result = client.files.content(output_file_id)
result.write_to_file(save_path)

print("Saved to:", save_path)

Saved to: /kaggle/working/teacher_batch_output.jsonl


In [37]:
path = "/kaggle/working/teacher_requests.jsonl"

with open(path, "r", encoding="utf-8") as f:
    first_req = json.loads(next(f))
    first_req = json.loads(next(f))

In [44]:
import json
import pandas as pd

OUTPUT_PATH = "/kaggle/working/teacher_batch_output.jsonl"

rows = []

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        custom_id = obj.get("custom_id")
        response = obj.get("response", {})
        status_code = response.get("status_code")
        body = response.get("body", {})

        raw_content = None
        parsed_output = None

        if status_code == 200:
            try:
                raw_content = body["choices"][0]["message"]["content"]
            except Exception:
                raw_content = None

            if raw_content is not None:
                try:
                    parsed_output = json.loads(raw_content)
                except json.JSONDecodeError:
                    parsed_output = None

        rows.append({
            "product_id": custom_id,
            "status_code": status_code,
            "raw_content": raw_content,
            "parsed_output": parsed_output,
        })

results_df = pd.DataFrame(rows)

print(results_df.shape)
results_df.head()

(1538, 4)


,product_id,status_code,raw_content,parsed_output
0,B0000C73CQ,200,"{""aspect_summaries"":{""kargo"":""Kargo hızlı ve s...",{'aspect_summaries': {'kargo': 'Kargo hızlı ve...
1,B00017028M,200,"{""aspect_summaries"":{""kargo"":""Siparişlerin geç...",{'aspect_summaries': {'kargo': 'Siparişlerin g...
2,B00019RA8U,200,"{""aspect_summaries"":{""kargo"":""Teslimatın hızlı...",{'aspect_summaries': {'kargo': 'Teslimatın hız...
3,B0002LXXC4,200,"{""aspect_summaries"":{""kargo"":""Teslimat hızlı v...",{'aspect_summaries': {'kargo': 'Teslimat hızlı...
4,B000AQVRUG,200,"{""aspect_summaries"":{""kargo"":""Ürün hızlı ve sa...",{'aspect_summaries': {'kargo': 'Ürün hızlı ve ...


In [45]:
import pandas as pd

flat_rows = []

for _, row in results_df.iterrows():
    parsed = row["parsed_output"] or {}
    aspect_summaries = parsed.get("aspect_summaries", {})

    flat_rows.append({
        "product_id": row["product_id"],
        "status_code": row["status_code"],
        "kargo": aspect_summaries.get("kargo"),
        "paketleme": aspect_summaries.get("paketleme"),
        "fiyat_performans": aspect_summaries.get("fiyat_performans"),
        "kalite": aspect_summaries.get("kalite"),
        "genel_memnuniyet": aspect_summaries.get("genel_memnuniyet"),
    })

summary_df = pd.DataFrame(flat_rows)

print(summary_df.shape)
summary_df.head()

(1538, 7)


,product_id,status_code,kargo,paketleme,fiyat_performans,kalite,genel_memnuniyet
0,B0000C73CQ,200,Kargo hızlı ve sorunsuz teslim edilmiş görünüyor.,Paketleme özenli ve ürün korunaklı şekilde ula...,Fiyatı uygun ve fiyatına göre avantajlı bulunmuş.,"Ürün kaliteli, orijinal ve beklentiyi karşılıyor.",Kullanıcılar üründen genel olarak memnun ve te...
1,B00017028M,200,Siparişlerin geç ulaştığı belirtilmiş.,Ürün güzel ambalajlanmış ve orijinal halde gön...,None,"Tuzun lezzeti artırdığı, yiyeceğe çıtırlık ve ...",Ürün güzel bulunmuş ve memnuniyet ifade edilmiş.
2,B00019RA8U,200,Teslimatın hızlı ve sorunsuz olduğu belirtiliyor.,Paketlemenin koruyucu ve düzenli olduğu ifade ...,Ürünün fiyatına göre avantajlı ve hesaplı oldu...,Pil kalitesinin iyi ve uzun ömürlü olduğu vurg...,Kullanıcılar üründen genel olarak memnun olduk...
3,B0002LXXC4,200,Teslimat hızlı ve zamanında yapılmış görünüyor.,Paketleme özenli ve ürünler sağlam şekilde ula...,Fiyatı uygun bulunmuş ve sunduğu değer olumlu ...,Pil kalitesi ve kullanım performansı genel ola...,Ürün genel olarak memnuniyet yaratmış ve tavsi...
4,B000AQVRUG,200,Ürün hızlı ve sağlam şekilde teslim edilmiş.,Ürün kutusuyla birlikte sağlam ve orijinal hal...,Fiyatına göre alınabilir ve tavsiye edilen bir...,"Saatin malzeme ve işçilik kalitesi beğenilmiş,...",Kullanıcılar üründen memnun kalmış ve gönül ra...


In [46]:
summary_df.to_csv("teacher_batch_output_flattened.csv")

In [44]:
import pandas as pd
import numpy as np

import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [45]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [46]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print("Loaded in 4-bit.")

2026-03-28 08:37:45,372 - httpx - INFO
Msg: HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"

2026-03-28 08:37:45,444 - httpx - INFO
Msg: HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"

2026-03-28 08:37:45,511 - httpx - INFO
Msg: HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.1-8B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"

2026-03-28 08:37:45,576 - httpx - INFO
Msg: HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.1-8B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"

2026-03-28 08:37:46,845 - httpx - INFO
Msg: HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.1-8B-Instruct "HTTP/1.1 200 OK"

2026-03-28 08:37:46,950 - httpx - INFO
Msg: HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-03-28 08:38:06,369 - httpx - INFO
Msg: HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"

Loaded in 4-bit.


In [47]:
import torch

def chat_with_system(system_message, user_message, max_new_tokens=250, temperature=0.2, top_p=0.9):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # sadece yeni üretilen kısmı al
    response = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

In [50]:
print(chat_with_system(first_req['body']['messages'][0]['content'], first_req['body']['messages'][1]['content']))

{
  "aspect_summaries": {
    "kargo": "Teslimat hızlı ve sorunsuz gerçekleşti.",
    "paketleme": null,
    "fiyat_performans": "Ürün uygun fiyatla satın alınabiliyor.",
    "kalite": "Ürün kaliteli ve hassasiyete iyi geliyor.",
    "genel_memnuniyet": "Ürün tüm beklentileri karşıladı ve memnun edici."
  }
}
